## Step 6: Data Distribution — Dirichlet (label-skew) & Reconstruction-Error (quality-skew) Non-IID

**Role of reconstruction error:** it is used **only to partition client data** (a data-quality Non-IID axis, Figure 14). It is *not* an input to CRTAT: Equation/Algorithm for CRTAT contain no reconstruction-error term.


In [ ]:
def split_dirichlet(X, y, n_clients, alpha=0.5, min_samples=150, seed=SEED):
    """
    Dirichlet-based Non-IID split — realistic heterogeneous FL.
    alpha: concentration parameter (lower = more skewed / non-IID)
    """
    rng = np.random.default_rng(seed)
    n_classes = len(np.unique(y))
    class_indices = {c: np.where(y == c)[0] for c in range(n_classes)}
    for c in class_indices: rng.shuffle(class_indices[c])

    client_indices = defaultdict(list)
    for c in range(n_classes):
        proportions = rng.dirichlet(alpha * np.ones(n_clients))
        proportions = proportions / proportions.sum()
        idx = class_indices[c]
        splits = (proportions * len(idx)).astype(int)
        splits[-1] = len(idx) - splits[:-1].sum()
        ptr = 0
        for cid, cnt in enumerate(splits):
            client_indices[cid].extend(idx[ptr:ptr+cnt].tolist())
            ptr += cnt

    splits_out = []
    for cid in range(n_clients):
        idx = np.array(client_indices[cid])
        if len(idx) < min_samples:
            extra = rng.choice(len(X), min_samples - len(idx), replace=True)
            idx   = np.concatenate([idx, extra])
        rng.shuffle(idx)
        splits_out.append((X[idx], y[idx]))
    return splits_out


def compute_reconstruction_error(X_raw, pca):
    """Per-sample PCA reconstruction error: ||x - PCA^-1(PCA(x))||_2.

    Low error  → sample lies close to the dataset's dominant PCA subspace
                 (a 'typical' / easily-represented sample).
    High error → sample is atypical / an outlier under the learned subspace.
    """
    X_proj  = pca.transform(X_raw)
    X_recon = pca.inverse_transform(X_proj)
    return np.linalg.norm(X_raw - X_recon, axis=1)


def split_reconstruction_error(X_raw, X_features, y, pca, n_clients,
                                n_bins=RECON_ERROR_BINS, alpha=RECON_ERROR_ALPHA,
                                min_samples=150, seed=SEED):
    """
    v7 NEW — Reconstruction-Error-based Non-IID split.

    Instead of partitioning samples by CLASS LABEL (as split_dirichlet does),
    this partitions by PCA RECONSTRUCTION-ERROR quantile bin, using the same
    Dirichlet-concentration mechanism. Each bin plays the role a 'class' plays
    in the label-based split, so clients end up with a Dirichlet-skewed mix of
    'easy' (low reconstruction-error, typical) and 'hard' (high reconstruction-
    error, atypical/outlier) samples.

    This models a DATA-QUALITY axis of heterogeneity that is complementary to,
    and independent of, label-skew — clients differ not just in *which classes*
    they hold but in how *typical* their local data is. This is directly
    relevant to poisoning detection, since poisoned/outlier updates often
    originate from atypical local data distributions.

    Returns:
        splits_out         : list of (X_client, y_client) per client
        client_mean_error  : array[n_clients] — mean reconstruction error per client
        errors             : array[len(X_raw)] — per-sample reconstruction error
    """
    rng = np.random.default_rng(seed)
    errors = compute_reconstruction_error(X_raw, pca)

    bin_edges = np.quantile(errors, np.linspace(0, 1, n_bins + 1))
    bin_edges[-1] += 1e-6
    bin_ids = np.digitize(errors, bin_edges[1:-1])   # values in [0, n_bins-1]

    bin_indices = {b: np.where(bin_ids == b)[0] for b in range(n_bins)}
    for b in bin_indices: rng.shuffle(bin_indices[b])

    client_indices = defaultdict(list)
    for b in range(n_bins):
        proportions = rng.dirichlet(alpha * np.ones(n_clients))
        proportions = proportions / proportions.sum()
        idx = bin_indices[b]
        splits = (proportions * len(idx)).astype(int)
        splits[-1] = len(idx) - splits[:-1].sum()
        ptr = 0
        for cid, cnt in enumerate(splits):
            client_indices[cid].extend(idx[ptr:ptr+cnt].tolist())
            ptr += cnt

    splits_out, client_mean_error = [], []
    for cid in range(n_clients):
        idx = np.array(client_indices[cid])
        if len(idx) < min_samples:
            extra = rng.choice(len(X_features), min_samples - len(idx), replace=True)
            idx   = np.concatenate([idx, extra])
        rng.shuffle(idx)
        splits_out.append((X_features[idx], y[idx]))
        client_mean_error.append(float(errors[idx].mean()))
    return splits_out, np.array(client_mean_error), errors


def inject_label_flip(y, from_lbl, to_lbl, fraction=POISON_FRAC, rng=None):
    """Flip `fraction` of samples with label `from_lbl` to `to_lbl`."""
    if rng is None: rng = np.random.default_rng(SEED)
    y2 = y.copy()
    mask = (y2 == from_lbl)
    if fraction < 1.0:
        flip_idx = np.where(mask)[0]
        n_keep = int(len(flip_idx) * (1 - fraction))
        keep   = rng.choice(flip_idx, n_keep, replace=False) if n_keep > 0 else np.array([])
        if len(keep) > 0: mask[keep] = False
    y2[mask] = to_lbl
    return y2

def inject_multi_label_flip(y, fraction=POISON_FRAC, rng=None):
    """v4: dual-target attack — flip both POISON_FROM→POISON_TO AND POISON_FROM_2→POISON_TO_2.
    More damaging than single-class flip; multi-target attacks are harder for
    gradient-distance defenses to catch by simple norm/magnitude."""
    y = inject_label_flip(y, POISON_FROM,   POISON_TO,   fraction=fraction, rng=rng)
    y = inject_label_flip(y, POISON_FROM_2, POISON_TO_2, fraction=fraction, rng=rng)
    return y

# ─── Dirichlet Non-IID split (label-skew) — all three datasets ──────
client_cifar    = split_dirichlet(Xf_cifar,    y_cifar,    NUM_CLIENTS, alpha=0.5)
client_cifar100 = split_dirichlet(Xf_cifar100, y_cifar100, NUM_CLIENTS, alpha=0.5)
client_mnist    = split_dirichlet(Xf_mnist,    y_mnist,    NUM_CLIENTS, alpha=0.5)

sizes_c   = [len(s[1]) for s in client_cifar]
sizes_c100= [len(s[1]) for s in client_cifar100]
sizes_m   = [len(s[1]) for s in client_mnist]

print(f'✅ Dirichlet Non-IID split complete (label-skew).')
print(f'   CIFAR-10  — min: {min(sizes_c)}, max: {max(sizes_c)}, '
      f'mean: {np.mean(sizes_c):.0f}, std: {np.std(sizes_c):.0f}')
print(f'   CIFAR-100 — min: {min(sizes_c100)}, max: {max(sizes_c100)}, '
      f'mean: {np.mean(sizes_c100):.0f}, std: {np.std(sizes_c100):.0f}')
print(f'   MNIST     — min: {min(sizes_m)}, max: {max(sizes_m)}, '
      f'mean: {np.mean(sizes_m):.0f}, std: {np.std(sizes_m):.0f}')
print(f'   Attack fraction = {POISON_FRAC:.0%} of eligible source-class labels')

# ─── Reconstruction-Error Non-IID split (data-quality skew) — v7 NEW ─
client_cifar_recon, err_mean_cifar, err_all_cifar = split_reconstruction_error(
    Xc_flat, Xf_cifar, y_cifar, pca_cifar, NUM_CLIENTS)
client_cifar100_recon, err_mean_cifar100, err_all_cifar100 = split_reconstruction_error(
    Xc100_flat, Xf_cifar100, y_cifar100, pca_cifar100, NUM_CLIENTS)
client_mnist_recon, err_mean_mnist, err_all_mnist = split_reconstruction_error(
    Xm_norm, Xf_mnist, y_mnist, pca_mnist, NUM_CLIENTS)

print(f'\n✅ Reconstruction-Error Non-IID split complete (data-quality skew).')
print(f'   CIFAR-10  — per-client mean recon-error range: '
      f'[{err_mean_cifar.min():.3f}, {err_mean_cifar.max():.3f}]')
print(f'   CIFAR-100 — per-client mean recon-error range: '
      f'[{err_mean_cifar100.min():.3f}, {err_mean_cifar100.max():.3f}]')
print(f'   MNIST     — per-client mean recon-error range: '
      f'[{err_mean_mnist.min():.3f}, {err_mean_mnist.max():.3f}]')
print(f'   ({RECON_ERROR_BINS} error-quantile bins, Dirichlet alpha={RECON_ERROR_ALPHA})')


✅ Dirichlet Non-IID split complete (label-skew).
   CIFAR-10  — min: 236, max: 1720, mean: 1000, std: 384
   CIFAR-100 — min: 736, max: 2285, mean: 1000, std: 266
   MNIST     — min: 171, max: 2066, mean: 1000, std: 443
   Attack fraction = 70% of eligible source-class labels

✅ Reconstruction-Error Non-IID split complete (data-quality skew).
   CIFAR-10  — per-client mean recon-error range: [2.738, 4.205]
   CIFAR-100 — per-client mean recon-error range: [2.784, 4.494]
   MNIST     — per-client mean recon-error range: [0.092, 0.138]
   (10 error-quantile bins, Dirichlet alpha=0.5)
